# 01. Данные, разбиение по времени, аудит признаков (этап 1)
**Вариант 5: прогноз банкротства компаний** — Kaggle «Company Bankruptcy Prediction» (Тайвань, `data.csv`).
Цель — `Bankrupt?` (банкротство в течение года, 0/1). Метрики варианта: Gini и полнота при фиксированной доле проверок.

**Контрольная точка (3–5 предложений).** Вывод приведён в конце ноутбука.

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
root = os.getcwd()                              # ищем корень проекта (где лежит config.yaml) — работает и в Colab
while not os.path.exists(os.path.join(root, "config.yaml")) and root != os.path.dirname(root):
    root = os.path.dirname(root)
sys.path.insert(0, root); os.chdir(root)
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30, "display.width", 160, "display.float_format", "{:.4f}".format)
from src import pipeline as P
c = P.context()

## 1. Данные: размер, пропуски, цель

In [ ]:
df, t = c["df"], c["target"]
print("строк:", len(df), "| признаков:", df.shape[1] - 2, "| пропусков всего:", int(df.isna().sum().sum()))
print(df[t].value_counts().rename("n").to_frame().assign(share=lambda x: x.n / len(df)))

## 2. Описание каждого признака
Тип, доля пропусков, число уникальных значений, диапазон и одномерный Gini на train. Экономический смысл признаков понятен из названий (все — коэффициенты годовой отчётности: рентабельность, ликвидность, долговая нагрузка, оборачиваемость, денежные потоки, рост). Колонка `status` показывает, оставлен ли признак после аудита.

In [ ]:
r1 = P.ensure("step1")
desc = pd.read_csv("artifacts/features_description.csv")
desc

## 3. Дата наблюдения и разбиение по времени
В датасете **нет колонки с датой**. Данные собраны за 1999–2009 гг. (Taiwan Economic Journal) и записаны в хронологическом порядке, поэтому **порядок строк используется как прокси времени**. Это главное ограничение работы (см. README и отчёт): проверка — доля банкротств монотонно снижается по порядку строк (рисунок ниже), т.е. порядок строк действительно несёт информацию о периоде.

Разбиение: train — первые 70%, validation — следующие 15%, test (out-of-time) — последние 15%. Test в ноутбуках 01–03 **не используется**.

In [ ]:
print(pd.read_csv("artifacts/split_summary.csv"))
P.show_fig("fig1_drift.png")

**Случайное разбиение для сравнения.** Одна и та же простая модель, оценённая на validation при разбиении по времени и при случайном разбиении, — демонстрация, почему случайное разбиение недопустимо (оценка получается на «перемешанных» периодах).

In [ ]:
r1["split_demo"]

## 4. Проверка утечки данных и аудит признаков
1. **Признаки «после события»**: просмотрены все 95 названий. Все показатели — коэффициенты отчётности за период ДО банкротства (рентабельность, ликвидность, долг, оборачиваемость, потоки). Признаков вида «сумма просрочки», «статус после» и т.п. нет → утечки по смыслу не найдено.
2. **Статистическая проверка**: одномерный Gini любого признака на train не должен быть подозрительно высоким (порог 0,90).
3. **Константы и дубли**: удаляем признаки, не несущие информации, и «двойников» (|rho| ≥ 0,98 по Спирмену), оставляя более сильного.
4. **Процесс**: все статистики (границы винзоризации, квантили для флага стресса, WoE-бины, пороги) считаются только на train.

Удалённые признаки и причины:

In [ ]:
pd.read_csv("artifacts/removed_features.csv")

In [ ]:
print("максимальный одномерный Gini:", round(r1["max_univariate_gini"], 3))
print("осталось сырых признаков:", r1["n_features_after_audit"])

**Вывод.** В выборке 6819 компаний и 220 банкротов (3,2%). Данные разделены по порядку строк на train (4773), validation (1023) и test (1023); test не использовался до этапа 4. Даты в данных нет, поэтому порядок строк — прокси времени; доля банкротов в целом падает с 4–9% в начале до 1–2% в конце, что это подтверждает. Утечки «после события» в данных нет; удалены 20 признаков: 1 константа и 19 дублей (корреляция не ниже 0,98).